In [9]:
import pandas as pd
import numpy as np

print("Python is working!")
print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)

Python is working!
Pandas version: 2.2.3
NumPy version: 2.1.3


In [10]:
import pandas as pd
import os

data_path = os.path.join(
    os.path.expanduser("~"),
    "OneDrive",
    "Desktop",
    "Network-Anomaly-Detection-and-Predictive-Intrusion-Monitoring-Platform",
    "data",
    "tuesday.csv"
)

print("File exists:", os.path.exists(data_path))

File exists: True


In [11]:
df = pd.read_csv(data_path, nrows=10000, low_memory=False)

print("Dataset loaded successfully!")
print("Number of rows and columns:", df.shape)

print("\nFirst 5 rows:")
display(df.head())

print("\nColumn names:")
print(df.columns.tolist())

print("\nAttack labels:")
print(df["Label"].value_counts())

Dataset loaded successfully!
Number of rows and columns: (10000, 89)

First 5 rows:


,Src IP dec,Src Port,Dst IP dec,Dst Port,Protocol,Timestamp,Flow Duration,Total Fwd Packet,Total Bwd packets,Total Length of Fwd Packet,...,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,ICMP Code,ICMP Type,Total TCP Flow Time,Label,Attempted Category
0,3232238085,49159,3232238083,445,6,53:44.4,90030854,10,10,8,...,57,29987508.67,3.559250e+04,30013373,29946916,-1,-1,90030854,BENIGN,-1
1,134610945,0,134219268,0,0,54:12.4,106007973,26,0,0,...,19,19816839.20,8.154881e+06,27220170,7234941,-1,-1,0,BENIGN,-1
2,3232238085,123,3232238083,123,17,54:32.2,64015367,4,4,192,...,139,64015127.00,0.000000e+00,64015127,64015127,-1,-1,0,BENIGN,-1
3,3232238083,60280,3232238081,53,17,55:07.6,46870,1,1,44,...,0,0.00,0.000000e+00,0,0,-1,-1,0,BENIGN,-1
4,3232238083,61995,3232238081,53,17,54:12.4,62958,1,1,48,...,0,0.00,0.000000e+00,0,0,-1,-1,0,BENIGN,-1



Column names:
['Src IP dec', 'Src Port', 'Dst IP dec', 'Dst Port', 'Protocol', 'Timestamp', 'Flow Duration', 'Total Fwd Packet', 'Total Bwd packets', 'Total Length of Fwd Packet', 'Total Length of Bwd Packet', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd RST Flags', 'Bwd RST Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Packet Length Min', 'Packet Length Max', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Fl

In [12]:
import numpy as np

# Convert labels into two classes:
# 0 = Normal traffic
# 1 = Attack traffic
df["Attack"] = (
    df["Label"].astype(str).str.strip() != "BENIGN"
).astype(int)

# Select numeric features only
X = df.select_dtypes(include=[np.number]).drop(
    columns=["Attempted Category"], errors="ignore"
)

# Remove columns that are not useful for training
X = X.drop(
    columns=["Src IP dec", "Dst IP dec"],
    errors="ignore"
)

# Replace infinite values and missing values
X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(0)

# Target variable
y = df["Attack"]

print("Features available:", X.shape[1])
print("\nTraffic class counts:")
print(y.value_counts().rename(index={0: "Normal", 1: "Attack"}))

print("\nMissing values remaining:", int(X.isna().sum().sum()))
print("Data preparation completed!")

Features available: 85

Traffic class counts:
Attack
Normal    9989
Attack      11
Name: count, dtype: int64

Missing values remaining: 0
Data preparation completed!


In [13]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

print("Training the model...")

# Create the model
model = DecisionTreeClassifier(
    max_depth=5,
    class_weight="balanced",
    random_state=42
)

# Train the model
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

print("\nModel training completed!")
print("Accuracy:", round(accuracy_score(y_test, y_pred) * 100, 2), "%")

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    labels=[0, 1],
    target_names=["Normal", "Attack"],
    zero_division=0
))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred, labels=[0, 1]))

Training the model...

Model training completed!
Accuracy: 100.0 %

Classification Report:
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00      2997
      Attack       1.00      1.00      1.00         3

    accuracy                           1.00      3000
   macro avg       1.00      1.00      1.00      3000
weighted avg       1.00      1.00      1.00      3000

Confusion Matrix:
[[2997    0]
 [   0    3]]


In [14]:
wednesday_path = os.path.join(
    os.path.dirname(data_path),
    "wednesday.csv"
)

df_wednesday = pd.read_csv(
    wednesday_path,
    nrows=30000,
    low_memory=False
)

print("Dataset shape:", df_wednesday.shape)
print("\nTraffic labels:")
print(df_wednesday["Label"].value_counts())

Dataset shape: (30000, 89)

Traffic labels:
Label
BENIGN                          22642
DoS Slowloris                    3724
DoS Slowloris - Attempted        1730
DoS Slowhttptest                 1234
DoS Slowhttptest - Attempted      670
Name: count, dtype: int64


In [15]:
# Convert labels into two classes
df_wednesday["Attack"] = (
    df_wednesday["Label"].astype(str).str.strip() != "BENIGN"
).astype(int)

# Select numeric features
X_wed = df_wednesday.select_dtypes(
    include=[np.number]
).drop(
    columns=["Attempted Category", "Src IP dec", "Dst IP dec"],
    errors="ignore"
)

# Handle invalid and missing values
X_wed = X_wed.replace([np.inf, -np.inf], np.nan)
X_wed = X_wed.fillna(0)

# Target variable
y_wed = df_wednesday["Attack"]

print("Features:", X_wed.shape[1])
print("\nTraffic class counts:")
print(y_wed.value_counts().rename(index={0: "Normal", 1: "Attack"}))

print("\nMissing values:", int(X_wed.isna().sum().sum()))
print("Wednesday data preparation completed!")

Features: 85

Traffic class counts:
Attack
Normal    22642
Attack     7358
Name: count, dtype: int64

Missing values: 0
Wednesday data preparation completed!


In [16]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# Split into training and testing data
X_train_wed, X_test_wed, y_train_wed, y_test_wed = train_test_split(
    X_wed,
    y_wed,
    test_size=0.30,
    random_state=42,
    stratify=y_wed
)

# Create the model
model_wed = DecisionTreeClassifier(
    max_depth=5,
    class_weight="balanced",
    random_state=42
)

# Train
model_wed.fit(X_train_wed, y_train_wed)

# Predict
y_pred_wed = model_wed.predict(X_test_wed)

# Evaluate
print("Wednesday model training completed!")
print("Accuracy:", round(accuracy_score(y_test_wed, y_pred_wed) * 100, 2), "%")

print("\nClassification Report:")
print(classification_report(
    y_test_wed,
    y_pred_wed,
    labels=[0, 1],
    target_names=["Normal", "Attack"],
    zero_division=0
))

print("Confusion Matrix:")
print(confusion_matrix(y_test_wed, y_pred_wed, labels=[0, 1]))

Wednesday model training completed!
Accuracy: 100.0 %

Classification Report:
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00      6793
      Attack       1.00      1.00      1.00      2207

    accuracy                           1.00      9000
   macro avg       1.00      1.00      1.00      9000
weighted avg       1.00      1.00      1.00      9000

Confusion Matrix:
[[6793    0]
 [   0 2207]]


## CICIDS2017 Network Threat Detection — Initial Model Training

### Objective
Build an initial machine-learning model to classify network traffic as normal or attack traffic using the CICIDS2017 dataset.

### Dataset
- Source: CICIDS2017
- Sample: First 30,000 rows from Wednesday's CSV file
- Total features available for training: 85 numeric features
- Normal traffic: 22,642 records
- Attack and attempted-attack traffic: 7,358 records

### Data Preprocessing
- Converted traffic labels into binary classes: Normal (0) and Attack (1).
- Selected numeric network traffic features.
- Removed source and destination IP address columns and the attempted-category column.
- Replaced infinite and missing feature values.

### Model
- Algorithm: Decision Tree Classifier
- Maximum tree depth: 5
- Class weighting: Balanced
- Train/test split: 70% training and 30% testing
- Random state: 42

### Initial Evaluation Results
- Test records: 9,000
- Accuracy: 100% on this test split
- Normal records correctly classified: 6,793
- Attack records correctly classified: 2,207
- False positives: 0
- False negatives: 0

### Conclusion and Limitations
The initial experiment successfully trained a decision tree to distinguish normal and attack traffic within the selected Wednesday sample. The observed 100% test accuracy requires further investigation and independent validation. Additional testing on a separate day or dataset is needed to assess generalization and identify possible feature leakage.

This experiment is an initial prototype, not a production-ready intrusion detection system.